# Voxel statistics

Port of `packages/niivue/examples/vox.stats.html`. Demonstrates asymmetric positive and negative statistical thresholds, outline width, clip planes, and cutaway rendering.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
from ipyniivue import NiiVue

BASE_URL = "https://niivue.github.io/mono"
VOLUMES = f"{BASE_URL}/volumes"
MESHES = f"{BASE_URL}/meshes"

def hex_to_rgba(value):
    value = value.lstrip("#")
    return [int(value[i : i + 2], 16) / 255 for i in (0, 2, 4)] + [1.0]

def planes_for(count):
    if count == 0:
        return [[2.0, 180, 20]]
    if count == 1:
        return [[0.1, 180, 20]]
    if count == 2:
        return [[0.1, 180, 20], [0.1, 0, -20]]
    if count == 3:
        return [[0.0, 90, 0], [0.0, 0, -20], [0.1, 0, -90]]
    if count == 4:
        return [[0.3, 270, 0], [0.3, 90, 0], [0.0, 180, 0], [0.1, 0, 0]]
    if count == 5:
        return [[0.4, 270, 0], [0.4, 90, 0], [0.4, 180, 0], [0.2, 0, 0], [0.1, 0, -90]]
    return [[0.4, 270, 0], [-0.1, 90, 0], [0.4, 180, 0], [0.2, 0, 0], [0.1, 0, -90], [0.3, 0, 90]]

nv = NiiVue(
    background_color=[0.1, 0.1, 0.1, 1.0],
    is_colorbar_visible=True,
    show_render=1,
    slice_type=3,
    backend="webgl2",
)

slice_type = widgets.Dropdown(
    options=[("Axial", 0), ("Coronal", 1), ("Sagittal", 2), ("A+C+S+R", 3), ("Render", 4)],
    value=3,
    description="Slice",
)
clip_count = widgets.Dropdown(options=list(range(7)), value=3, description="Planes")
cutaway = widgets.Checkbox(value=True, description="Cutaway")
clip_color = widgets.Dropdown(
    options=[("Transparent", 0.0), ("Translucent", 0.3), ("Shading", -0.2)],
    value=-0.2,
    description="Clip color",
)
positive = widgets.FloatRangeSlider(value=[3.0, 6.0], min=0.0, max=15.0, step=0.1, description="Positive")
negative = widgets.FloatRangeSlider(value=[-6.0, -3.0], min=-15.0, max=0.0, step=0.1, description="Negative")
alpha_mode = widgets.Dropdown(
    options=[("Range only", 0), ("Transparent subthreshold", 1), ("Translucent subthreshold", 2)],
    value=0,
    description="Alpha",
)
outline = widgets.IntSlider(value=0, min=0, max=4, step=1, description="Outline")
background = widgets.ColorPicker(value="#191920", description="Background")

def update_slice(change):
    nv.slice_type = change["new"]

def update_planes(_change=None):
    nv.set_clip_planes(planes_for(clip_count.value))

def update_cutaway(change):
    nv.is_clip_plane_cutaway = change["new"]

def update_clip_color(change):
    nv.clip_plane_color = [1, 1, 1, change["new"]]

def update_thresholds(_change=None):
    pos_min, pos_max = positive.value
    neg_max, neg_min = negative.value
    nv.set_volume(
        1,
        {
            "calMin": pos_min,
            "calMax": pos_max,
            "calMinNeg": neg_min,
            "calMaxNeg": neg_max,
            "colormapType": alpha_mode.value,
        },
    )

def update_outline(change):
    nv.volume_outline_width = 0.25 * change["new"]

def update_background(change):
    nv.background_color = hex_to_rgba(change["new"])

slice_type.observe(update_slice, names="value")
clip_count.observe(update_planes, names="value")
cutaway.observe(update_cutaway, names="value")
clip_color.observe(update_clip_color, names="value")
positive.observe(update_thresholds, names="value")
negative.observe(update_thresholds, names="value")
alpha_mode.observe(update_thresholds, names="value")
outline.observe(update_outline, names="value")
background.observe(update_background, names="value")

controls = widgets.VBox([
    widgets.HBox([slice_type, clip_count, cutaway, clip_color, background]),
    widgets.HBox([alpha_mode, outline]),
    positive,
    negative,
])
display(widgets.VBox([controls, nv]))

nv.load_volumes([
    {"url": f"{VOLUMES}/mni152.nii.gz", "isColorbarVisible": False, "calMin": 30, "calMax": 80},
    {"url": f"{VOLUMES}/spmMotor.nii.gz", "colormap": "redyell", "colormapNegative": "winter"},
])
update_thresholds()
update_planes()
update_cutaway({"new": cutaway.value})
update_clip_color({"new": clip_color.value})
